In [6]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 1e-4 #from the paper for this chi
chi = 10
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 3 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 3, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[4], accepted_elements, _ = fix_discrete_gauge(traj[4]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array 

Newton iterations below (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [7]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA1 = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method
deltaA2 = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:3
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    #deltaA[i] = newton_correction(A[i], 5, accepted_elements[i], gilt_pars);
    deltaA1[i], deltaA2[i] = newton_correction_with_iterations_fixed(A[i], 20, accepted_elements[i], 
        gilt_pars, order = 2);
    println("||deltaA1[i]||,||deltaA2[i]|| = ", norm(deltaA1[i]), " ",norm(deltaA2[i]))
    A[i+1] = A[i] + deltaA1[i]
end

i=1
||R(A[i])-A[i]||= 0.03024339787576557
Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  21 eigenvalues converged
│ *  norm of residuals = (6.228197930259696e-60, 5.549631986426844e-44, 1.95224276621912e-44, 3.481541155836676e-34, 7.379139595126595e-28, 1.0442064191303322e-27, 1.0442064191303322e-27, 1.3783414224808256e-27, 1.3783414224808256e-27, 1.9024138486073615e-26, 1.9024138486073615e-26, 8.162519845302373e-23, 8.162519845302373e-23, 2.0482317769593127e-20, 1.4266392197591705e-20, 1.4266392197591705e-20, 3.3286402352175985e-17, 9.87165263700826e-18, 1.1118881930922448e-14, 8.153160896078879e-14, 3.005681816543505e-13)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
1.9972967396772736 + 0.0im
-0.9162211034209682 + 0.0im
-0.9131233959046187 + 0.0im
0.442168623950921 + 0.0im
-0.33454358288194347 + 0.0im
-3.748325760623743e-5 + 0.31204004961865567im
-3.748325760623743e-5 - 0.31204004961865567im
-0.07264178475609515 + 0.2939389871146371im
-0.07264178475609515 - 0.2939389871146371im
0.07262977905538819 + 0.2938191213851633im
0.07262977905538819 - 0.2938191213851633im
0.2413290252334142 + 0.031685555435610885im
0.2413290252334142 - 0.031685555435610885im
0.21720905967644577 + 0.0im
3.8879548577247055e-5 + 0.19690906334639802im
3.8879548577247055e-5 - 0.19690906334639802im
0.18159823386874616 + 0.0im
-0.17183524551250093 + 0.0im
-0.14896655041637966 + 0.0im
-0.1410599350840249 + 0.0im
-0.1212448262294764 + 0.0im
||deltaA1[i]||,||deltaA2[i]|| = 0.032560939801335655 0.0007519259646605495
i=2
||R(A[i])-A[i]||= 0.0014312551438814267
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 31, (2, "N") => 1, (2, "W") =

┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  20 eigenvalues converged
│ *  norm of residuals = (1.724267864219776e-61, 6.520096105469853e-45, 5.2574233681120356e-45, 4.62955136255642e-34, 2.494051208944795e-30, 8.171246896009986e-28, 8.171246896009986e-28, 6.088675246418264e-27, 6.088675246418264e-27, 3.536151395672457e-27, 3.536151395672457e-27, 1.156658167953041e-21, 1.156658167953041e-21, 2.045867489493041e-20, 3.6434725376221675e-20, 3.6434725376221675e-20, 1.894885133399577e-19, 5.891236431799176e-17, 3.24228860405205e-16, 1.988196875403549e-15)
└ *  number of operations = 52


EIGENVALUES (INITIAL):
2.007321917960751 + 0.0im
-0.9420104715168879 + 0.0im
-0.9377686386855859 + 0.0im
0.44450699322954595 + 0.0im
-0.3666078134171989 + 0.0im
-1.3295237647375068e-5 + 0.31935940928746387im
-1.3295237647375068e-5 - 0.31935940928746387im
-0.06228836746730777 + 0.2965001170281171im
-0.06228836746730777 - 0.2965001170281171im
0.06227890108173048 + 0.29649987834750996im
0.06227890108173048 - 0.29649987834750996im
0.23379974841143575 + 0.05092204219390001im
0.23379974841143575 - 0.05092204219390001im
0.22578387386146515 + 0.0im
2.1175814814072882e-5 + 0.19371152059285304im
2.1175814814072882e-5 - 0.19371152059285304im
-0.18877757763844427 + 0.0im
0.18573003271765928 + 0.0im
-0.15897491949970144 + 0.0im
-0.14988825894192528 + 0.0im
||deltaA1[i]||,||deltaA2[i]|| = 0.0014225994373930045 0.0002790657914519535
i=3
||R(A[i])-A[i]||= 0.00020031423358149718
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (

┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  20 eigenvalues converged
│ *  norm of residuals = (9.728331569995298e-59, 2.18462908630494e-43, 2.92746294056591e-44, 1.0077878990979883e-33, 7.424415531475543e-31, 3.507232935045019e-26, 3.507232935045019e-26, 3.241286442049367e-27, 3.241286442049367e-27, 6.663917565698188e-27, 6.663917565698188e-27, 3.102040888164116e-22, 3.102040888164116e-22, 1.3289556068257604e-20, 3.059276455822636e-20, 3.059276455822636e-20, 5.706547866184175e-19, 6.07286176943005e-17, 8.481029319834233e-16, 3.0652448606624344e-14)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
2.007438029801575 + 0.0im
-0.9411035372323965 + 0.0im
-0.9363202200093794 + 0.0im
0.4443164283455617 + 0.0im
-0.36344658661415236 + 0.0im
-2.310705048501113e-6 + 0.3184843760830902im
-2.310705048501113e-6 - 0.3184843760830902im
-0.06277255335078342 + 0.29611288704796673im
-0.06277255335078342 - 0.29611288704796673im
0.06277972133017931 + 0.296108305680464im
0.06277972133017931 - 0.296108305680464im
0.23417838549739722 + 0.050071650733930294im
0.23417838549739722 - 0.050071650733930294im
0.22516597452079423 + 0.0im
2.0705891364093477e-5 + 0.1942721623396539im
2.0705891364093477e-5 - 0.1942721623396539im
-0.18922081104139898 + 0.0im
0.18555749475472605 + 0.0im
-0.15872837690254554 + 0.0im
-0.14932946106638387 + 0.0im
||deltaA1[i]||,||deltaA2[i]|| = 0.0002662455808577691 9.856669102703951e-6


In [59]:
[A[1].sects[(0,0,0,0)][1,1,1,k] for k in 1:5]

5-element Vector{Float64}:
  0.522485325196873
 -0.033576763776410355
  1.9616523475617108e-5
  0.0009738712889111517
 -0.0014535226288984624

In [60]:
[A[1].sects[(0,0,0,0)][1,1,k,1] for k in 1:5]

5-element Vector{Float64}:
  0.522485325196873
 -0.03394577484519207
  8.89451283973256e-6
  0.0010640011018901601
 -0.0017764333291880657

In [61]:
[A[1].sects[(0,1,0,1)][1,1,1,k] for k in 1:5]

5-element Vector{Float64}:
 0.20430388454118534
 4.7977025223324214e-5
 0.02394000496786203
 0.0010356523081185322
 8.630948099597806e-7